# Gold fact: `fact_region_population`

## Purpose

Publish authoritative 2024 Census Table C population per official region. The dashboard divides regional budget and flood exposure by it.

## Grain

One region per population reference year and source.

- Uniqueness: `region_key`, `reference_year`, `source_name`.
- Key: `region_population_fact_key` is `XXHASH64` of the region key, reference year, and source name.

## What Gold reuses and does not redo

Every measure comes from `02-silver.silver_region_population`. Silver already summed only accepted barangay rows and kept a missing region population as `NULL`. Gold only adds `region_key`. It never adds province or city subtotals, and never turns a missing population into zero.

Central Office and other key `0` records have no population. Ratios must leave them out.

## Safe reruns

`CREATE TABLE IF NOT EXISTS` and `MERGE` on the deterministic key. Rows are never deleted.

## 1. Confirm upstream safety and select versions

**What:** check that `silver_region_population` is one validated run and that the Gold region dimension exists.

**Why:** Gold must not publish population from an unsafe Silver run, or point at a region key that is missing.

**Protects:** every check stops the notebook before any write.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `03-gold`;

DECLARE OR REPLACE VARIABLE population_rule_version STRING DEFAULT 'gold-fact-region-population-v1';

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT NAMED_STRUCT(
        'region', psgc_region_code, 'year', reference_year, 'source', source_name
    )),
    'STOP: silver_region_population must be nonempty, one run, and unique per region, year, and source'
)
FROM `02-silver`.silver_region_population;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(dq.action = 'stop' AND dq.status IN ('FAIL', 'ERROR')) = 0,
    'STOP: selected Silver region population validation is missing or unsafe'
)
FROM `04-validation`.silver_dq_results AS dq
WHERE dq.table_name IN ('silver_region_population')
  AND dq.pipeline_run_id = (SELECT MAX(population.run_id) FROM `02-silver`.silver_region_population AS population);

SELECT ASSERT_TRUE(
    (SELECT COUNT(*) FROM `03-gold`.dim_region WHERE region_key = 0) = 1
    AND (SELECT COUNT(*) FROM `03-gold`.dim_flood_susceptibility WHERE flood_susceptibility_key = 0) = 1,
    'STOP: run the Gold dimension notebooks 00 to 04 before the facts'
);

CREATE OR REPLACE TEMPORARY VIEW census_load_lineage AS
SELECT run_id AS source_ingest_run_id, source_version, source_modified_at,
       COALESCE(completed_at, started_at) AS source_load_ts
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS audit_rank
    FROM `01-bronze`.load_log
    WHERE table_name = 'census_2024_table_c'
      AND status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
)
WHERE audit_rank = 1;

## 2. Create the table

**What:** create `03-gold.fact_region_population` with the column names, types, and order in `docs/gold_model.md`.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS `03-gold`.fact_region_population (
    region_population_fact_key BIGINT NOT NULL COMMENT 'Surrogate key. XXHASH64 of the region key, reference year, and source name.',
    region_key BIGINT NOT NULL COMMENT 'Region of this population. Joins to dim_region.region_key.',
    reference_year INT COMMENT 'Census reference year. The reference date is July 1 of this year.',
    population_count BIGINT COMMENT 'Authoritative Table C population of the region. NULL when no accepted barangay row exists, never zero. Do not sum across years or sources.',
    source_name STRING COMMENT 'Population source, the PSA 2024 Census Table C.',
    is_primary_source BOOLEAN COMMENT 'True for the authoritative population source. Use only primary rows in ratios.',
    population_match_status STRING COMMENT 'MATCHED_PRIMARY_POPULATION, or MISSING_PRIMARY_POPULATION when the region has no accepted barangay rows.',
    source_row_count BIGINT COMMENT 'Number of accepted Table C barangay rows summed into population_count.',
    run_id STRING COMMENT 'Deterministic Gold run ID. Hash of the Silver population run and the Gold rule version.',
    source_load_ts TIMESTAMP COMMENT 'Bronze load time of the Table C snapshot behind this row.'
)
USING DELTA
COMMENT 'Gold population fact. One official region per reference year and source, from authoritative 2024 Census Table C. Central Office and unknown records have no population.';

## 3. Build and merge the rows

**What:** attach `region_key` from `dim_region` and merge by key.

**Why:** the region join uses both the PSGC code and the PSGC version, so a future PSGC release never joins to the wrong region member.

**Expected validation:** every Silver row resolves to a region, and keys are unique.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW fact_region_population_source AS
WITH resolved AS (
    SELECT
        region.region_key,
        population.reference_year,
        population.population_count,
        population.source_name,
        population.is_primary_source,
        population.population_match_status,
        population.source_row_count,
        SHA2(CONCAT_WS('|', population.run_id, population_rule_version), 256) AS run_id,
        census.source_load_ts
    FROM `02-silver`.silver_region_population AS population
    LEFT JOIN `03-gold`.dim_region AS region
        ON population.psgc_region_code = region.psgc_region_code
       AND population.psgc_version <=> region.psgc_version
    LEFT JOIN census_load_lineage AS census
        ON population.source_ingest_run_id = census.source_ingest_run_id
)
SELECT
    XXHASH64(CONCAT_WS(
        '|', 'REGION_POPULATION', CAST(region_key AS STRING), CAST(reference_year AS STRING), source_name
    )) AS region_population_fact_key,
    *
FROM resolved;

SELECT ASSERT_TRUE(
    COUNT(*) = (SELECT COUNT(*) FROM `02-silver`.silver_region_population)
    AND COUNT_IF(region_key IS NULL OR region_key = 0) = 0
    AND COUNT(*) = COUNT(DISTINCT region_population_fact_key)
    AND COUNT_IF(source_load_ts IS NULL) = 0,
    'STOP: every population row must resolve to one official region with a unique key and a load time'
)
FROM fact_region_population_source;

MERGE INTO `03-gold`.fact_region_population AS target
USING fact_region_population_source AS source
ON target.region_population_fact_key = source.region_population_fact_key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

# Summary

Created or refreshed `03-gold.fact_region_population` with one row per official region, reference year, and source. Population comes unchanged from Silver. The query below reconciles the Gold total with Silver. Next: `06_gold_fact_region_flood_exposure`.

In [ ]:
%sql
SELECT
    (SELECT COUNT(*) FROM `03-gold`.fact_region_population) AS gold_rows,
    (SELECT SUM(population_count) FROM `03-gold`.fact_region_population) AS gold_population,
    (SELECT SUM(population_count) FROM `02-silver`.silver_region_population) AS silver_population,
    (SELECT COUNT_IF(population_count IS NULL) FROM `03-gold`.fact_region_population) AS regions_without_population;